# Name: M Danish Zaheer Awan
# Roll no: 25280092

# Assignment 3 - Part 1

In [1]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Any, Dict, List
import great_expectations as gx

## Folders path defined

In [2]:
root = Path.cwd().resolve().parent
data = root / "Data"

## Functions Defined

In [3]:
def build_batch_from_dataframe(df: pd.DataFrame, source_name: str, asset_name: str, batch_name: str):
    context = gx.get_context()
    data_source = context.data_sources.add_pandas(name=source_name)
    data_asset = data_source.add_dataframe_asset(name=asset_name)
    batch_definition = data_asset.add_batch_definition_whole_dataframe(batch_name)
    batch = batch_definition.get_batch(batch_parameters={"dataframe": df})
    return context, batch_definition, batch

In [4]:
def gx_result_to_dict(result: Any) -> Dict[str, Any]:
    if isinstance(result, dict):
        return result

    for method_name in ("to_json_dict", "as_dict", "model_dump"):
        method = getattr(result, method_name, None)
        if callable(method):
            try:
                converted = method()
                if isinstance(converted, dict):
                    return converted
            except Exception:
                pass

    for method_name in ("to_json", "model_dump_json", "json"):
        method = getattr(result, method_name, None)
        if callable(method):
            try:
                raw = method()
                if isinstance(raw, str):
                    return json.loads(raw)
            except Exception:
                pass

    try:
        return dict(result)
    except Exception:
        return {"repr": repr(result)}

In [5]:
def summarize_single_expectation_result(result_dict: Dict[str, Any]) -> Dict[str, Any]:
    config = result_dict.get("expectation_config", {})
    res = result_dict.get("result", {})
    kwargs = config.get("kwargs", {})

    return {
        "expectation_type": config.get("type") or config.get("expectation_type", "unknown"),
        "column": kwargs.get("column"),
        "success": result_dict.get("success"),
        "unexpected_count": res.get("unexpected_count"),
        "unexpected_percent": res.get("unexpected_percent"),
        "partial_unexpected_list": res.get("partial_unexpected_list", []),
    }

### Task-1 

In [6]:
# has been implemented successfully

## Task 2 - Data Quality Report with Great Expectations on Synthetic Data

In [7]:
gx_project_path = Path("Task_2_deliverables")
gx_project_path.mkdir(exist_ok=True)

output_path = Path(gx_project_path / "synthetic_csv_reports")
output_path.mkdir(exist_ok=True)

expectations_path = gx_project_path / "expectations_definitions"
expectations_path.mkdir(exist_ok=True)

validations_path = gx_project_path / "validations_reports"
validations_path.mkdir(exist_ok=True)

### Clean Data EDA

In [8]:
df = pd.read_csv(data / "clean_synthetic_dataset.csv", low_memory=False)
df["sale_ts"] = pd.to_datetime(df["sale_ts"], errors="coerce")

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nData Types:")
print(df.dtypes)
print("\nFirst 5 Rows:")
display(df.head())

Shape: (1000000, 9)

Columns:
['sale_id', 'customer_id', 'product_id', 'sale_ts', 'quantity', 'amount', 'channel', 'currency', 'notes']

Data Types:
sale_id                            int64
customer_id                        int64
product_id                         int64
sale_ts        datetime64[ns, UTC+05:00]
quantity                           int64
amount                           float64
channel                           object
currency                          object
notes                            float64
dtype: object

First 5 Rows:


,sale_id,customer_id,product_id,sale_ts,quantity,amount,channel,currency,notes
0,1,23620,13664,2025-09-24 11:15:42.204402+05:00,4,77.48,online,USD,NaN
1,2,93717,10536,2026-02-21 07:55:42.204402+05:00,2,27.84,store,EUR,NaN
2,3,140749,4467,2025-05-16 11:12:42.204402+05:00,9,180.83,online,GBP,NaN
3,4,110518,169,2026-01-15 18:26:42.204402+05:00,2,106.37,partner,GBP,NaN
4,5,1957,1721,2026-01-08 07:52:42.204402+05:00,3,209.91,mobile_app,GBP,NaN


In [9]:
df.describe()

,sale_id,customer_id,product_id,quantity,amount,notes
count,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,0.0
mean,500000.500000,95114.445918,9011.774129,3.301856,199.658248,NaN
std,288675.278933,60162.059754,6234.456559,4.732620,782.898796,NaN
min,1.000000,1.000000,1.000000,1.000000,2.040000,NaN
25%,250000.750000,42020.000000,3337.000000,2.000000,56.480000,NaN
50%,500000.500000,94859.500000,8881.000000,3.000000,108.420000,NaN
75%,750000.250000,147334.000000,14459.000000,4.000000,166.050000,NaN
max,1000000.000000,200000.000000,20000.000000,219.000000,41533.860000,NaN


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 9 columns):
 #   Column       Non-Null Count    Dtype                    
---  ------       --------------    -----                    
 0   sale_id      1000000 non-null  int64                    
 1   customer_id  1000000 non-null  int64                    
 2   product_id   1000000 non-null  int64                    
 3   sale_ts      1000000 non-null  datetime64[ns, UTC+05:00]
 4   quantity     1000000 non-null  int64                    
 5   amount       1000000 non-null  float64                  
 6   channel      1000000 non-null  object                   
 7   currency     1000000 non-null  object                   
 8   notes        0 non-null        float64                  
dtypes: datetime64[ns, UTC+05:00](1), float64(2), int64(4), object(2)
memory usage: 68.7+ MB


In [11]:
def build_expectations_1() -> List[Any]:
    expectations: List[Any] = [
        gx.expectations.ExpectColumnToExist(column="sale_id", severity="critical"),
        gx.expectations.ExpectColumnToExist(column="customer_id", severity="critical"),
        gx.expectations.ExpectColumnToExist(column="product_id", severity="critical"),
        gx.expectations.ExpectColumnToExist(column="sale_ts", severity="critical"),
        gx.expectations.ExpectColumnToExist(column="quantity", severity="critical"),
        gx.expectations.ExpectColumnToExist(column="amount", severity="critical"),
        gx.expectations.ExpectColumnToExist(column="channel", severity="critical"),
        gx.expectations.ExpectColumnToExist(column="currency", severity="critical"),

        gx.expectations.ExpectColumnValuesToBeOfType(column="sale_id", type_="int64", severity="critical"),
        gx.expectations.ExpectColumnValuesToBeOfType(column="customer_id", type_="int64", severity="critical"),
        gx.expectations.ExpectColumnValuesToBeOfType(column="product_id", type_="int64", severity="critical"),
        gx.expectations.ExpectColumnValuesToBeOfType(column="quantity", type_="int64", severity="critical"),
        gx.expectations.ExpectColumnValuesToBeOfType(column="amount", type_="float64", severity="warning"),
        gx.expectations.ExpectColumnValuesToBeOfType(column="channel", type_="object", severity="warning"),
        gx.expectations.ExpectColumnValuesToBeOfType(column="currency", type_="object", severity="warning"),

        gx.expectations.ExpectColumnValuesToNotBeNull(column="sale_id", mostly=1.0, severity="critical"),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="customer_id", mostly=1.0, severity="critical"),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="product_id", mostly=1.0, severity="critical"),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="sale_ts", mostly=1.0, severity="critical"),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="quantity", mostly=1.0, severity="critical"),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="amount", mostly=1.0, severity="critical"),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="channel", mostly=1.0, severity="critical"),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="currency", mostly=1.0, severity="critical"),

        gx.expectations.ExpectColumnValuesToBeUnique(column="sale_id", severity="critical"),

        gx.expectations.ExpectColumnValuesToBeBetween(column="sale_id", min_value=1, max_value=1000000, severity="critical"),
        gx.expectations.ExpectColumnValuesToBeBetween(column="customer_id", min_value=1, max_value=200000, severity="critical"),
        gx.expectations.ExpectColumnValuesToBeBetween(column="product_id", min_value=1, max_value=20000, severity="critical"),
        gx.expectations.ExpectColumnValuesToBeBetween(column="quantity", min_value=1, max_value=219, severity="critical"),
        gx.expectations.ExpectColumnValuesToBeBetween(column="amount", min_value=2.04, max_value=41533.86, severity="warning"),

        gx.expectations.ExpectTableRowCountToBeBetween(min_value=900000, max_value=1000000, severity="warning"),

        gx.expectations.ExpectColumnValuesToBeInSet(
            column="channel",
            value_set=["mobile_app", "online", "partner", "store"],
            severity="warning",
        ),
        gx.expectations.ExpectColumnValuesToBeInSet(
            column="currency",
            value_set=["EUR", "GBP", "PKR", "USD"],
            severity="critical",
        ),

        gx.expectations.ExpectColumnValuesToBeBetween(
            column="sale_ts",
            min_value="2025-03-26 19:23:42.204402+05:00",
            max_value="2026-03-26 19:22:42.204402+05:00",
            severity="warning",
        ),

        gx.expectations.ExpectColumnQuantileValuesToBeBetween(
            column="quantity",
            quantile_ranges={
                "quantiles": [0.25, 0.50, 0.75],
                "value_ranges": [[2, 2], [3, 3], [4, 4]],
            },
            severity="warning",
        ),
    ]
    return expectations

In [12]:
context, batch_definition, batch = build_batch_from_dataframe(df, "task2_synthetic_clean_source", "task2_synthetic_clean_asset", "task2_synthetic_clean_batch")
        
print(f"GX version: {getattr(gx, '__version__', 'unknown')}")
print(f"Context type: {type(context).__name__}")
print("Batch created successfully.")

GX version: 1.15.1
Context type: EphemeralDataContext
Batch created successfully.


In [13]:
expectations    = build_expectations_1()
print(f"Number of expectations defined: {len(expectations)}")
        
for idx, expectation in enumerate(expectations, start=1):
    print(f"{idx:02d}. {type(expectation).__name__}")

Number of expectations defined: 34
01. ExpectColumnToExist
02. ExpectColumnToExist
03. ExpectColumnToExist
04. ExpectColumnToExist
05. ExpectColumnToExist
06. ExpectColumnToExist
07. ExpectColumnToExist
08. ExpectColumnToExist
09. ExpectColumnValuesToBeOfType
10. ExpectColumnValuesToBeOfType
11. ExpectColumnValuesToBeOfType
12. ExpectColumnValuesToBeOfType
13. ExpectColumnValuesToBeOfType
14. ExpectColumnValuesToBeOfType
15. ExpectColumnValuesToBeOfType
16. ExpectColumnValuesToNotBeNull
17. ExpectColumnValuesToNotBeNull
18. ExpectColumnValuesToNotBeNull
19. ExpectColumnValuesToNotBeNull
20. ExpectColumnValuesToNotBeNull
21. ExpectColumnValuesToNotBeNull
22. ExpectColumnValuesToNotBeNull
23. ExpectColumnValuesToNotBeNull
24. ExpectColumnValuesToBeUnique
25. ExpectColumnValuesToBeBetween
26. ExpectColumnValuesToBeBetween
27. ExpectColumnValuesToBeBetween
28. ExpectColumnValuesToBeBetween
29. ExpectColumnValuesToBeBetween
30. ExpectTableRowCountToBeBetween
31. ExpectColumnValuesToBeInSet


In [14]:
suite = gx.ExpectationSuite(name="general_synthetic_expectation_suite")
suite = context.suites.add(suite)

for expectation in expectations: suite.add_expectation(expectation)

print(f"Expectation Suite name  : {suite.name}")
print(f"Expectations in suite   :  {len(suite.expectations)}")

Expectation Suite name  : general_synthetic_expectation_suite
Expectations in suite   :  34


In [15]:
suite_json_path = expectations_path / "synthetic_expectation_suite.json"

with suite_json_path.open("w", encoding="utf-8") as f:
    json.dump(suite.to_json_dict(), f, indent=2, default=str)

print(f"Saved expectation suite.")

Saved expectation suite.


In [16]:
summaries: List[Dict[str, Any]] = []
for expectation in expectations:
    result      = batch.validate(expectation)
    result_dict = gx_result_to_dict(result)
    summary     = summarize_single_expectation_result(result_dict)
    summaries.append(summary)

    """print(f"\nExpectation   : {summary['expectation_type']}")
    print(f"Success         : {summary['success']}")
    print(f"Unexpected      : {summary['unexpected_count']}")
            
    if summary["partial_unexpected_list"]:
        print(f"Examples    : {summary['partial_unexpected_list']}")"""

summary_df          = pd.DataFrame(summaries)
summary_csv_path    = output_path / "clean_validation_summary.csv"
summary_df.to_csv(summary_csv_path, index=False)

"""print("\nCompact validation summary:")
print(summary_df)"""
print(f"\nSaved summary.")

Calculating Metrics: 100%|██████████| 4/4 [00:00<00:00, 300.63it/s] 


Saved summary.


In [17]:
validation_definition = context.validation_definitions.add(
            gx.core.validation_definition.ValidationDefinition(name = "synthetic_validation",
                                                            data = batch_definition,
                                                            suite= suite,))

suite_result        = validation_definition.run(batch_parameters={"dataframe": df})
suite_result_dict   = gx_result_to_dict(suite_result)

suite_result_path   = validations_path / "clean_validation_result.json"
with suite_result_path.open("w", encoding="utf-8") as f:
    json.dump(suite_result_dict, f, indent=2, default=str)

print("Full suite validation finished.")
print("Overall success:", suite_result_dict.get("success"))
print(f"Saved suite result JSON.") 

Calculating Metrics: 100%|██████████| 96/96 [00:01<00:00, 85.55it/s] 

Full suite validation finished.
Overall success: True
Saved suite result JSON.


## For corrupted data

In [18]:
corrupted_df = pd.read_csv(data / "corrupted_synthetic_dataset.csv", low_memory=False)
corrupted_df["sale_ts"] = pd.to_datetime(corrupted_df["sale_ts"], errors="coerce", utc=True)

print("Shape:", corrupted_df.shape)
display(corrupted_df.head())

Shape: (1040203, 9)


,sale_id,customer_id,product_id,sale_ts,quantity,amount,channel,currency_raw,notes
0,2,93717.0,10536.0,2026-02-21 02:55:42.204402+00:00,2,27.84,store,EUR,NaN
1,3,140749.0,4467.0,2025-05-16 06:12:42.204402+00:00,9,180.83,online,GBP,NaN
2,5,1957.0,1721.0,2026-01-08 02:52:42.204402+00:00,3,209.91,mobile_app,GBP,NaN
3,6,191772.0,90.0,2025-04-05 21:32:42.204402+00:00,4,152.59,store,EUR,NaN
4,8,112959.0,1827.0,2025-07-15 18:18:42.204402+00:00,4,190.72,partner,PKR,NaN


In [19]:
context, batch_definition, batch = build_batch_from_dataframe(corrupted_df, "task2_synthetic_corrupted_source", "task2_synthetic_corrupted_asset", "task2_synthetic_corrupted_batch")

print(f"GX version: {getattr(gx, '__version__', 'unknown')}")
print(f"Context type: {type(context).__name__}")
print("Batch created successfully.")

GX version: 1.15.1
Context type: EphemeralDataContext
Batch created successfully.


In [20]:
expectations_2 = build_expectations_1()
print(f"Number of expectations defined: {len(expectations_2)}")

for idx, expectation in enumerate(expectations_2, start=1):
    print(f"{idx:02d}. {type(expectation).__name__}")

Number of expectations defined: 34
01. ExpectColumnToExist
02. ExpectColumnToExist
03. ExpectColumnToExist
04. ExpectColumnToExist
05. ExpectColumnToExist
06. ExpectColumnToExist
07. ExpectColumnToExist
08. ExpectColumnToExist
09. ExpectColumnValuesToBeOfType
10. ExpectColumnValuesToBeOfType
11. ExpectColumnValuesToBeOfType
12. ExpectColumnValuesToBeOfType
13. ExpectColumnValuesToBeOfType
14. ExpectColumnValuesToBeOfType
15. ExpectColumnValuesToBeOfType
16. ExpectColumnValuesToNotBeNull
17. ExpectColumnValuesToNotBeNull
18. ExpectColumnValuesToNotBeNull
19. ExpectColumnValuesToNotBeNull
20. ExpectColumnValuesToNotBeNull
21. ExpectColumnValuesToNotBeNull
22. ExpectColumnValuesToNotBeNull
23. ExpectColumnValuesToNotBeNull
24. ExpectColumnValuesToBeUnique
25. ExpectColumnValuesToBeBetween
26. ExpectColumnValuesToBeBetween
27. ExpectColumnValuesToBeBetween
28. ExpectColumnValuesToBeBetween
29. ExpectColumnValuesToBeBetween
30. ExpectTableRowCountToBeBetween
31. ExpectColumnValuesToBeInSet


In [21]:
suite_2 = gx.ExpectationSuite(name="general_synthetic_expectation_suite_corrupted")
suite_2 = context.suites.add(suite_2)

In [22]:
for expectation in expectations_2:
    suite_2.add_expectation(expectation)

print(f"Expectation Suite name  : {suite_2.name}")
print(f"Expectations in suite   :  {len(suite_2.expectations)}")

Expectation Suite name  : general_synthetic_expectation_suite_corrupted
Expectations in suite   :  34


In [23]:
summaries: List[Dict[str, Any]] = []
for expectation in expectations_2:
    result      = batch.validate(expectation)
    result_dict = gx_result_to_dict(result)
    summary     = summarize_single_expectation_result(result_dict)
    summaries.append(summary)

    """print(f"\nExpectation   : {summary['expectation_type']}")
    print(f"Success         : {summary['success']}")
    print(f"Unexpected      : {summary['unexpected_count']}")
            
    if summary["partial_unexpected_list"]:
        print(f"Examples    : {summary['partial_unexpected_list']}")"""

summary_df          = pd.DataFrame(summaries)
summary_csv_path    = output_path / "corrupted_validation_summary.csv"
summary_df.to_csv(summary_csv_path, index=False)

"""print("\nCompact validation summary:")
print(summary_df)"""
print(f"\nSaved summary")

Calculating Metrics: 100%|██████████| 4/4 [00:00<00:00, 269.07it/s] 


Saved summary


In [24]:
validation_definition = context.validation_definitions.add(
            gx.core.validation_definition.ValidationDefinition(name = "synthetic_validation_corrupted",
                                                            data = batch_definition,
                                                            suite= suite_2,))

suite_result        = validation_definition.run(batch_parameters={"dataframe": corrupted_df})
suite_result_dict   = gx_result_to_dict(suite_result)

suite_result_path   = validations_path / "corrupted_validation_result.json"
with suite_result_path.open("w", encoding="utf-8") as f:
    json.dump(suite_result_dict, f, indent=2, default=str)

print("Full suite validation finished.")
print("Overall success:", suite_result_dict.get("success"))
print(f"Saved suite result JSON.") 

Calculating Metrics:  88%|████████▊ | 84/96 [00:03<00:00, 27.52it/s] 

Full suite validation finished.
Overall success: False
Saved suite result JSON.


## Task 3 - Data Quality Report with Great Expectations on PakWheels Dataset

# Part 1 - Task 3: Great Expectations on PakWheels Dataset


In [25]:
gx_project_dir = Path("Task_3_deliverables")
gx_project_dir.mkdir(exist_ok=True)

output_dir = gx_project_dir / "PakWheels_csv_report"
output_dir.mkdir(exist_ok=True)

expectations_dir = gx_project_dir / "expectations_definitions"
expectations_dir.mkdir(exist_ok=True)

validations_dir = gx_project_dir / "validations_report"
validations_dir.mkdir(exist_ok=True)

In [26]:
df_pakwheels = pd.read_csv(data / "PakWheelsData.csv", low_memory=False)

print("Shape:", df_pakwheels.shape)
print("\nColumns:")
print(df_pakwheels.columns.tolist())
print("\nData Types:")
print(df_pakwheels.dtypes)
print("\nFirst 5 Rows:")
display(df_pakwheels.head())

Shape: (62302, 14)

Columns:
['addref', 'city', 'assembly', 'body', 'make', 'model', 'year', 'engine', 'transmission', 'fuel', 'color', 'registered', 'mileage', 'price']

Data Types:
addref            int64
city             object
assembly         object
body             object
make             object
model            object
year            float64
engine          float64
transmission     object
fuel             object
color            object
registered       object
mileage           int64
price           float64
dtype: object

First 5 Rows:


,addref,city,assembly,body,make,model,year,engine,transmission,fuel,color,registered,mileage,price
0,7642697,Islamabad,NaN,Compact SUV,KIA,Sorento,2021.0,3500.0,Automatic,Petrol,NaN,Islamabad,54654,9300000.0
1,7909608,Sadiqabad,Imported,Hatchback,Daihatsu,Mira,2020.0,660.0,Automatic,Petrol,White,Punjab,10000,3700000.0
2,7929224,Peshawar,Imported,Hatchback,Toyota,Vitz,2018.0,1000.0,Automatic,Petrol,Silver,Islamabad,123000,4150000.0
3,7832366,Lahore,NaN,Sedan,Toyota,Corolla,2019.0,1600.0,Automatic,Petrol,White,Lahore,105000,4850000.0
4,7866725,Islamabad,NaN,Sedan,Toyota,Corolla,2022.0,1600.0,Automatic,Petrol,White,Islamabad,6500,6600000.0


In [27]:
df_pakwheels.describe()

,addref,year,engine,mileage,price
count,6.230200e+04,58516.000000,62299.000000,62302.000000,6.183000e+04
mean,7.809772e+06,2012.843991,1406.731887,91043.031476,3.886141e+06
std,2.619455e+05,7.497370,706.209678,89279.502257,5.452661e+06
min,4.478110e+05,1990.000000,3.000000,1.000000,1.100000e+05
25%,7.806020e+06,2007.000000,1000.000000,34000.000000,1.450000e+06
50%,7.865742e+06,2015.000000,1300.000000,80000.000000,2.700000e+06
75%,7.910367e+06,2019.000000,1600.000000,123456.000000,4.500000e+06
max,7.943741e+06,2022.000000,15000.000000,1000000.000000,1.650000e+08


In [28]:
df_pakwheels.describe(include=object)

,city,assembly,body,make,model,transmission,fuel,color,registered
count,62302,19374,55211,62302,62302,62302,61593,61110,62302
unique,287,1,21,66,408,2,3,376,115
top,Lahore,Imported,Sedan,Toyota,Corolla,Automatic,Petrol,White,Islamabad
freq,13363,19374,24176,19869,10219,34270,56550,17121,15152


In [29]:
df_pakwheels.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 62302 entries, 0 to 62301
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   addref        62302 non-null  int64  
 1   city          62302 non-null  object 
 2   assembly      19374 non-null  object 
 3   body          55211 non-null  object 
 4   make          62302 non-null  object 
 5   model         62302 non-null  object 
 6   year          58516 non-null  float64
 7   engine        62299 non-null  float64
 8   transmission  62302 non-null  object 
 9   fuel          61593 non-null  object 
 10  color         61110 non-null  object 
 11  registered    62302 non-null  object 
 12  mileage       62302 non-null  int64  
 13  price         61830 non-null  float64
dtypes: float64(3), int64(2), object(9)
memory usage: 6.7+ MB


In [30]:
def build_expectations_2() -> List[Any]:
    expectations: List[Any] = [
        gx.expectations.ExpectColumnToExist(column="addref", severity="warning"),
        gx.expectations.ExpectColumnToExist(column="city", severity="critical"),
        gx.expectations.ExpectColumnToExist(column="assembly", severity="warning"),
        gx.expectations.ExpectColumnToExist(column="body", severity="critical"),
        gx.expectations.ExpectColumnToExist(column="make", severity="warning"),
        gx.expectations.ExpectColumnToExist(column="model", severity="warning"),
        gx.expectations.ExpectColumnToExist(column="year", severity="critical"),
        gx.expectations.ExpectColumnToExist(column="engine", severity="critical"),
        gx.expectations.ExpectColumnToExist(column="transmission", severity="critical"),
        gx.expectations.ExpectColumnToExist(column="fuel", severity="critical"),
        gx.expectations.ExpectColumnToExist(column="color", severity="warning"),
        gx.expectations.ExpectColumnToExist(column="registered", severity="critical"),
        gx.expectations.ExpectColumnToExist(column="mileage", severity="critical"),
        gx.expectations.ExpectColumnToExist(column="price", severity="critical"),


        gx.expectations.ExpectColumnValuesToBeOfType(column="addref", type_="int64", severity="warning"),
        gx.expectations.ExpectColumnValuesToBeOfType(column="city", type_="object", severity="critical"),
        gx.expectations.ExpectColumnValuesToBeOfType(column="assembly", type_="object", severity="warning"),
        gx.expectations.ExpectColumnValuesToBeOfType(column="body", type_="object", severity="critical"),
        gx.expectations.ExpectColumnValuesToBeOfType(column="make", type_="object", severity="warning"),
        gx.expectations.ExpectColumnValuesToBeOfType(column="model", type_="object", severity="warning"),
        gx.expectations.ExpectColumnValuesToBeOfType(column="year", type_="float64", severity="critical"),
        gx.expectations.ExpectColumnValuesToBeOfType(column="engine", type_="float64", severity="critical"),
        gx.expectations.ExpectColumnValuesToBeOfType(column="transmission", type_="object", severity="critical"),
        gx.expectations.ExpectColumnValuesToBeOfType(column="fuel", type_="object", severity="critical"),
        gx.expectations.ExpectColumnValuesToBeOfType(column="color", type_="object", severity="critical"),
        gx.expectations.ExpectColumnValuesToBeOfType(column="registered", type_="object", severity="critical"),
        gx.expectations.ExpectColumnValuesToBeOfType(column="mileage", type_="int64", severity="critical"),
        gx.expectations.ExpectColumnValuesToBeOfType(column="price", type_="float64", severity="critical"),

        gx.expectations.ExpectColumnValuesToNotBeNull(column="addref", mostly=1.0, severity="warning"),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="city", mostly=1.0, severity="critical"),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="assembly", mostly=1.0, severity="warning"),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="body", mostly=1.0, severity="critical"),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="make", mostly=1.0, severity="warning"),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="model", mostly=1.0, severity="warning"),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="year", mostly=1.0, severity="critical"),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="engine", mostly=1.0, severity="critical"),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="transmission", mostly=1.0, severity="critical"),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="fuel", mostly=1.0, severity="critical"),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="color", mostly=1.0, severity="warning"),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="registered", mostly=1.0, severity="critical"),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="mileage", mostly=1.0, severity="critical"),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="price", mostly=1.0, severity="critical"),

        gx.expectations.ExpectColumnValuesToBeUnique(column="addref", severity="critical"),

        gx.expectations.ExpectColumnValuesToBeBetween(column="year", min_value=1990, max_value=2026, severity="warning"),
        gx.expectations.ExpectColumnValuesToBeBetween(column="engine", min_value=600, severity="warning"),
        gx.expectations.ExpectColumnValuesToBeBetween(column="mileage", min_value=100, max_value=500000, severity="warning"),
        gx.expectations.ExpectColumnValuesToBeBetween(column="price", min_value=500000, max_value=30000000, severity="warning"),

        gx.expectations.ExpectTableRowCountToBeBetween(min_value=50000, severity="critical"),

        gx.expectations.ExpectColumnValuesToBeInSet(
            column="transmission",
            value_set=["Automatic", "Manual"],
            severity="critical",
        ),
        gx.expectations.ExpectColumnValuesToBeInSet(
            column="fuel",
            value_set=["Petrol", "Diesel", "Hybrid"],
            severity="critical",
        ),
        gx.expectations.ExpectColumnValuesToBeInSet(
            column="body",
            value_set=["Sedan", "Hatchback", "SUV", "Crossover", "Mini Van", "Compact sedan",
                "Double Cabin", "MPV", "Van", "Micro Van", "Pick Up", "Compact SUV", "Station Wagon", "Coupe",
                "Truck", "High Roof", "Convertible", "Single Cabin", "Off-Road Vehicles", "Compact hatchback",
                "Mini Vehicles"], mostly=0.8, severity="warning",
        ),

        gx.expectations.ExpectColumnQuantileValuesToBeBetween(
            column="mileage",
            quantile_ranges={
                "quantiles": [0.25, 0.50, 0.75],
                "value_ranges": [[30000, 45000], [50000, 85000], [90000, 130000]],
            },
            severity="warning",
        ),
        gx.expectations.ExpectColumnQuantileValuesToBeBetween(
            column="price",
            quantile_ranges={
                "quantiles": [0.25, 0.50, 0.75],
                "value_ranges": [[1300000, 1600000], [1700000, 2900000], [3000000, 4700000]],
            },
            severity="warning",
        ),
    ]
    return expectations

In [31]:
context, batch_definition, batch = build_batch_from_dataframe(df_pakwheels, "task3_pakwheels_source", "task3_pakwheels_asset", "task3_pakwheels_batch")

print(f"GX version: {getattr(gx, '__version__', 'unknown')}")
print(f"Context type: {type(context).__name__}")

GX version: 1.15.1
Context type: EphemeralDataContext


In [32]:
expectations2    = build_expectations_2()
print(f"Number of expectations defined: {len(expectations2)}")
        
for idx, expectation in enumerate(expectations2, start=1):
    print(f"{idx:02d}. {type(expectation).__name__}")

Number of expectations defined: 53
01. ExpectColumnToExist
02. ExpectColumnToExist
03. ExpectColumnToExist
04. ExpectColumnToExist
05. ExpectColumnToExist
06. ExpectColumnToExist
07. ExpectColumnToExist
08. ExpectColumnToExist
09. ExpectColumnToExist
10. ExpectColumnToExist
11. ExpectColumnToExist
12. ExpectColumnToExist
13. ExpectColumnToExist
14. ExpectColumnToExist
15. ExpectColumnValuesToBeOfType
16. ExpectColumnValuesToBeOfType
17. ExpectColumnValuesToBeOfType
18. ExpectColumnValuesToBeOfType
19. ExpectColumnValuesToBeOfType
20. ExpectColumnValuesToBeOfType
21. ExpectColumnValuesToBeOfType
22. ExpectColumnValuesToBeOfType
23. ExpectColumnValuesToBeOfType
24. ExpectColumnValuesToBeOfType
25. ExpectColumnValuesToBeOfType
26. ExpectColumnValuesToBeOfType
27. ExpectColumnValuesToBeOfType
28. ExpectColumnValuesToBeOfType
29. ExpectColumnValuesToNotBeNull
30. ExpectColumnValuesToNotBeNull
31. ExpectColumnValuesToNotBeNull
32. ExpectColumnValuesToNotBeNull
33. ExpectColumnValuesToNotBeNu

In [33]:
suite_3 = gx.ExpectationSuite(name="pakwheels_expectation_suite")
suite_3 = context.suites.add(suite_3)

for expectation in expectations2:
    suite_3.add_expectation(expectation)

In [34]:
suite_json_path = expectations_dir / "pakwheels_expectation_suite.json"
with suite_json_path.open("w", encoding="utf-8") as f:
    json.dump(suite_3.to_json_dict(), f, indent=2, default=str)

print(f"Saved expectation suite.")

Saved expectation suite.


In [35]:
summaries: List[Dict[str, Any]] = []
for expectation in expectations2:
    result      = batch.validate(expectation)
    result_dict = gx_result_to_dict(result)
    summary     = summarize_single_expectation_result(result_dict)
    summaries.append(summary)

    """print(f"\nExpectation   : {summary['expectation_type']}")
    print(f"Success         : {summary['success']}")
    print(f"Unexpected      : {summary['unexpected_count']}")
            
    if summary["partial_unexpected_list"]:
        print(f"Examples    : {summary['partial_unexpected_list']}")"""

summary_df          = pd.DataFrame(summaries)
summary_csv_path = output_dir / "pakwheels_validation_summary.csv"
summary_df.to_csv(summary_csv_path, index=False)

"""print("\nCompact validation summary:")
print(summary_df)"""
print(f"\nSaved summary.")

Calculating Metrics: 100%|██████████| 4/4 [00:00<00:00, 855.41it/s] 


Saved summary.


In [36]:
validation_definition = context.validation_definitions.add(
            gx.core.validation_definition.ValidationDefinition(name = "pakwheels_validation",
                                                            data = batch_definition,
                                                            suite= suite_3,))

suite_result        = validation_definition.run(batch_parameters={"dataframe": df_pakwheels})
suite_result_dict   = gx_result_to_dict(suite_result)

validation_result_path = validations_dir / "pakwheels_validation_result.json"
with validation_result_path.open("w", encoding="utf-8") as f:
    json.dump(suite_result_dict, f, indent=2, default=str)

print("Full suite validation finished.")
print("Overall success:", suite_result_dict.get("success"))
print(f"Saved suite result JSON.") 

Calculating Metrics: 100%|██████████| 124/124 [00:00<00:00, 604.44it/s]


Full suite validation finished.
Overall success: False
Saved suite result JSON.


## Task 4 - Data Cleaning and Re-validation

In [37]:
gx_project_dir = Path("Task_4_deliverables")
gx_project_dir.mkdir(exist_ok=True)

summary_dir = gx_project_dir / "Summary_Reports"
summary_dir.mkdir(exist_ok=True)

validations_dir = gx_project_dir / "Validation_Reports"
validations_dir.mkdir(exist_ok=True)

## For Synthetic_Data

In [38]:
df = pd.read_csv(data / "cleaned_data" / "corrupted_synthetic_dataset_cleaned.csv", parse_dates=["sale_ts"], low_memory=False)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
display(df.head())

print("\nNull counts:")
display(df.isnull().sum().sort_values(ascending=False))

print("\nDtypes:")
print(df.dtypes)

Shape: (901802, 8)

Columns:
['sale_id', 'customer_id', 'product_id', 'sale_ts', 'quantity', 'amount', 'channel', 'currency']


,sale_id,customer_id,product_id,sale_ts,quantity,amount,channel,currency
0,2,93717,10536,2026-02-21 02:55:42.204402+00:00,2,27.84,store,EUR
1,3,140749,4467,2025-05-16 06:12:42.204402+00:00,9,180.83,online,GBP
2,5,1957,1721,2026-01-08 02:52:42.204402+00:00,3,209.91,mobile_app,GBP
3,6,191772,90,2025-04-05 21:32:42.204402+00:00,4,152.59,store,EUR
4,8,112959,1827,2025-07-15 18:18:42.204402+00:00,4,190.72,partner,PKR



Null counts:


sale_id        0
customer_id    0
product_id     0
sale_ts        0
quantity       0
amount         0
channel        0
currency       0
dtype: int64


Dtypes:
sale_id                      int64
customer_id                  int64
product_id                   int64
sale_ts        datetime64[ns, UTC]
quantity                     int64
amount                     float64
channel                     object
currency                    object
dtype: object


In [39]:
print("Final shape:", df.shape)

print("\nRemaining null counts:")
display(df.isnull().sum())

print("\nRemaining dtypes:")
print(df.dtypes)

display(df.head())

Final shape: (901802, 8)

Remaining null counts:


sale_id        0
customer_id    0
product_id     0
sale_ts        0
quantity       0
amount         0
channel        0
currency       0
dtype: int64


Remaining dtypes:
sale_id                      int64
customer_id                  int64
product_id                   int64
sale_ts        datetime64[ns, UTC]
quantity                     int64
amount                     float64
channel                     object
currency                    object
dtype: object


,sale_id,customer_id,product_id,sale_ts,quantity,amount,channel,currency
0,2,93717,10536,2026-02-21 02:55:42.204402+00:00,2,27.84,store,EUR
1,3,140749,4467,2025-05-16 06:12:42.204402+00:00,9,180.83,online,GBP
2,5,1957,1721,2026-01-08 02:52:42.204402+00:00,3,209.91,mobile_app,GBP
3,6,191772,90,2025-04-05 21:32:42.204402+00:00,4,152.59,store,EUR
4,8,112959,1827,2025-07-15 18:18:42.204402+00:00,4,190.72,partner,PKR


In [40]:
context, batch_definition, batch = build_batch_from_dataframe(df, "task4_synthetic_source", "task4_synthetic_asset", "task4_synthetic_batch")

print(f"GX version: {getattr(gx, '__version__', 'unknown')}")
print(f"Context type: {type(context).__name__}")
print("Batch created successfully.")


GX version: 1.15.1
Context type: EphemeralDataContext
Batch created successfully.


In [41]:
suite_json_path = Path("Task_2_deliverables") / "expectations_definitions" / "synthetic_expectation_suite.json"

with suite_json_path.open("r", encoding="utf-8") as f:
    suite_dict = json.load(f)


print("Loaded suite JSON successfully.")
print("Suite name:", suite_dict.get("name"))
print("Expectation count:", len(suite_dict.get("expectations", [])))

Loaded suite JSON successfully.
Suite name: general_synthetic_expectation_suite
Expectation count: 34


In [42]:
loaded_suite = gx.ExpectationSuite(
    name=suite_dict.get("name", "synthetic_expectation_suite"),
    expectations=suite_dict.get("expectations", []),
    suite_parameters=suite_dict.get("suite_parameters"),
    meta=suite_dict.get("meta"),
    notes=suite_dict.get("notes"),
)


print("Suite object rebuilt successfully.")
print("Suite name:", loaded_suite.name)
print("Expectation count:", len(loaded_suite.expectations))

Suite object rebuilt successfully.
Suite name: general_synthetic_expectation_suite
Expectation count: 34


In [43]:
loaded_suite = context.suites.add(loaded_suite)

print("Suite registered in GX context successfully.")
print("Suite name:", loaded_suite.name)


validation_definition = context.validation_definitions.add(
    gx.core.validation_definition.ValidationDefinition(
        name="synthetic_validation_from_json",
        data=batch_definition,
        suite=loaded_suite,
    )
)

print("Validation definition created successfully.")

Suite registered in GX context successfully.
Suite name: general_synthetic_expectation_suite
Validation definition created successfully.


In [44]:
suite_result = validation_definition.run(batch_parameters={"dataframe": df})
suite_result_dict = gx_result_to_dict(suite_result)

print("Validation completed.")
print("Overall success:", suite_result_dict.get("success"))

validation_result_path = validations_dir / "Synthethic_validation_result.json"

with validation_result_path.open("w", encoding="utf-8") as f:
    json.dump(suite_result_dict, f, indent=2, default=str)

print(f"Saved validation result JSON.")

Calculating Metrics: 100%|██████████| 96/96 [00:00<00:00, 100.34it/s]

Validation completed.
Overall success: True
Saved validation result JSON.


In [45]:
summaries = []

for result_dict in suite_result_dict.get("results", []):
    summary = summarize_single_expectation_result(result_dict)
    summaries.append(summary)

summary_df = pd.DataFrame(summaries)
summary_csv_path = summary_dir / "Synthethic_validation_summary.csv"
summary_df.to_csv(summary_csv_path, index=False)

print(f"Saved summary CSV.")

Saved summary CSV.


# For PakWheels Data

In [46]:
# 1. Load CSV
df = pd.read_csv(data / "cleaned_data" / "PakWheelsData_cleaned.csv")

# 4. Shape and head
print("\n")
print("Shape:", df.shape)
df.head()



Shape: (54755, 14)


,addref,city,assembly,body,make,model,year,engine,transmission,fuel,color,registered,mileage,price
0,7642697,Islamabad,Unknown,Compact SUV,KIA,Sorento,2021.0,3500.0,Automatic,Petrol,Unknown,Islamabad,54654,9300000.0
1,7909608,Sadiqabad,Imported,Hatchback,Daihatsu,Mira,2020.0,660.0,Automatic,Petrol,White,Punjab,10000,3700000.0
2,7929224,Peshawar,Imported,Hatchback,Toyota,Vitz,2018.0,1000.0,Automatic,Petrol,Silver,Islamabad,123000,4150000.0
3,7832366,Lahore,Unknown,Sedan,Toyota,Corolla,2019.0,1600.0,Automatic,Petrol,White,Lahore,105000,4850000.0
4,7866725,Islamabad,Unknown,Sedan,Toyota,Corolla,2022.0,1600.0,Automatic,Petrol,White,Islamabad,6500,6600000.0


In [47]:
context, batch_definition, batch = build_batch_from_dataframe(df, "task4_pakwheels_source", "task4_pakwheels_asset", "task4_pakwheels_batch")

print(f"GX version: {getattr(gx, '__version__', 'unknown')}")
print(f"Context type: {type(context).__name__}")
print("Batch created successfully.")

GX version: 1.15.1
Context type: EphemeralDataContext
Batch created successfully.


In [48]:
suite_json_path = Path("Task_3_deliverables") / "expectations_definitions" / "pakwheels_expectation_suite.json"

with suite_json_path.open("r", encoding="utf-8") as f:
    suite_dict = json.load(f)


print("Loaded suite JSON successfully.")
print("Suite name:", suite_dict.get("name"))
print("Expectation count:", len(suite_dict.get("expectations", [])))

Loaded suite JSON successfully.
Suite name: pakwheels_expectation_suite
Expectation count: 53


In [49]:
loaded_suite = gx.ExpectationSuite(
    name=suite_dict.get("name", "pakwheels_expectation_suite"),
    expectations=suite_dict.get("expectations", []),
    suite_parameters=suite_dict.get("suite_parameters"),
    meta=suite_dict.get("meta"),
    notes=suite_dict.get("notes"),
)


print("Suite object rebuilt successfully.")
print("Suite name:", loaded_suite.name)
print("Expectation count:", len(loaded_suite.expectations))

Suite object rebuilt successfully.
Suite name: pakwheels_expectation_suite
Expectation count: 53


In [50]:
loaded_suite = context.suites.add(loaded_suite)

print("Suite registered in GX context successfully.")
print("Suite name:", loaded_suite.name)


validation_definition = context.validation_definitions.add(
    gx.core.validation_definition.ValidationDefinition(
        name="pakwheels_validation",
        data=batch_definition,
        suite=loaded_suite,
    )
)

print("Validation definition created successfully.")

Suite registered in GX context successfully.
Suite name: pakwheels_expectation_suite
Validation definition created successfully.


In [51]:
suite_result = validation_definition.run(batch_parameters={"dataframe": df})
suite_result_dict = gx_result_to_dict(suite_result)

print("Validation completed.")
print("Overall success:", suite_result_dict.get("success"))

validation_result_path = validations_dir / "PakWheels_validation_result.json"

with validation_result_path.open("w", encoding="utf-8") as f:
    json.dump(suite_result_dict, f, indent=2, default=str)

print(f"Saved validation result JSON.")

Calculating Metrics: 100%|██████████| 124/124 [00:00<00:00, 876.97it/s]


Validation completed.
Overall success: True
Saved validation result JSON.


In [52]:
summaries = []

for result_dict in suite_result_dict.get("results", []):
    summary = summarize_single_expectation_result(result_dict)
    summaries.append(summary)

summary_df = pd.DataFrame(summaries)
summary_csv_path = summary_dir / "Pakwheels_validation_summary.csv"
summary_df.to_csv(summary_csv_path, index=False)

print(f"Saved summary CSV0.")

Saved summary CSV0.
